# Order Anomaly Detection Experiments — Isolation Forest + MLflow

**Purpose.** This notebook trains the current anomaly model (**Isolation Forest**) on the current
dataset and logs every run to **MLflow** so the team can tune hyperparameters and compare results.
It detects **order-level** anomalies — individual orders that are operationally unusual (very high
payment, heavy products, slow delivery, expensive freight, poor reviews).

**How to use it**
1. Select the project's **`.venv` (Python 3.11)** kernel — the only environment with
   `sklearn` / `mlflow` installed.
2. Edit the values in the **`HYPERPARAMS`** cell (marked ★ EDIT THESE).
3. Run all cells. Each run is logged to MLflow as a new entry.
4. Compare runs in the MLflow UI:
   ```
   .venv\Scripts\python -m mlflow ui --backend-store-uri sqlite:///mlflow.db
   ```
   then open http://localhost:5000 and look at the **`olist_anomaly_detection`** experiment.

> **Unsupervised model:** there are no ground-truth anomaly labels, so we compare configurations
> using the **anomaly rate** and **separation metrics** (how different the flagged orders are from
> normal ones). This notebook is **self-contained** and mirrors `ml_engine/anomaly_detection.py`.
> With the default hyperparameters it reproduces **2,315 anomalies (2.00%)**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder

import mlflow
import mlflow.sklearn

## Paths & MLflow setup

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Data Cleaning + EDA").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "Data Cleaning + EDA" / "cleaned_master_df.parquet"
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("olist_anomaly_detection")

print("Project root :", PROJECT_ROOT)
print("Data path    :", DATA_PATH, "(exists:", DATA_PATH.exists(), ")")
print("MLflow store :", MLFLOW_DB)

## Load data

In [ ]:
master = pd.read_parquet(DATA_PATH)
print(f"Loaded {master.shape[0]:,} rows x {master.shape[1]} columns")
master.head()

## Feature engineering
Builds the exact **29-feature** matrix the model expects (mirrors `_build_features` in
`ml_engine/anomaly_detection.py`): date parts, seller-to-customer distance, median/`"Unknown"`
fills, label-encoded categoricals, and three derived ratios. **Do not change the feature order** —
the model contract depends on it. (Feature engineering is a fair thing to experiment with too, but
keep the 29-column layout consistent within a run.)

In [ ]:
FEATURE_ORDER = [
    "order_status", "customer_state", "order_item_id", "price", "freight_value",
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "product_category", "seller_state", "payment_sequential", "payment_type",
    "payment_installments", "payment_value", "review_score", "delivery_days",
    "purchase_year", "purchase_month", "purchase_day", "purchase_weekday",
    "purchase_hour", "distance_km", "total_cost", "payment_per_installment",
    "freight_ratio",
]
CATEGORICAL_COLS = ["order_status", "customer_state", "product_category",
                    "seller_state", "payment_type"]


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


d = master.copy()

# date parts
ts = pd.to_datetime(d["order_purchase_timestamp"], errors="coerce")
d["purchase_year"] = ts.dt.year
d["purchase_month"] = ts.dt.month
d["purchase_day"] = ts.dt.day
d["purchase_weekday"] = ts.dt.weekday
d["purchase_hour"] = ts.dt.hour

# seller <-> customer distance
d["distance_km"] = haversine_km(d["customer_lat"], d["customer_lng"],
                                d["seller_lat"], d["seller_lng"])

# median-fill base numeric features
base_numeric = [c for c in FEATURE_ORDER if c not in CATEGORICAL_COLS
                and c not in ("total_cost", "payment_per_installment", "freight_ratio")]
for col in base_numeric:
    if col in d.columns:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        d[col] = d[col].fillna(d[col].median())

# categorical fill + label encode
label_encoders = {}
for col in CATEGORICAL_COLS:
    d[col] = d[col].astype(str).fillna("Unknown").replace("nan", "Unknown")
    le = LabelEncoder()
    d[col] = le.fit_transform(d[col])
    label_encoders[col] = le

# derived features (after fills -> no NaN)
d["payment_installments"] = d["payment_installments"].replace(0, 1)
d["total_cost"] = d["price"] + d["freight_value"]
d["payment_per_installment"] = d["payment_value"] / d["payment_installments"]
d["freight_ratio"] = d["freight_value"] / (d["price"] + 1)

X_df = d[FEATURE_ORDER].copy()
print("Feature matrix:", X_df.shape, "| NaNs:", int(X_df.isnull().sum().sum()))
X_df.head()

## ★ HYPERPARAMETERS — EDIT THESE
These are the **current production defaults**. Change any value, then **Run All** to log a new
MLflow run.

| Param | Meaning |
|---|---|
| `n_estimators` | number of trees in the forest |
| `contamination` | expected fraction of anomalies (drives the flag threshold) |
| `random_state` | seed for reproducibility (keep at 42 to match the script) |

In [ ]:
HYPERPARAMS = {
    "n_estimators":  100,
    "contamination": 0.02,
    "random_state":  42,
}

## Scale, train & log to MLflow
Standard-scales the features, fits the Isolation Forest, and logs params + metrics + model. The
**separation metrics** (mean of each business field for normal vs anomaly) let you judge whether a
configuration is flagging *meaningfully* unusual orders.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_df)

with mlflow.start_run() as run:
    mlflow.log_params(HYPERPARAMS)

    model = IsolationForest(**HYPERPARAMS)
    model.fit(X)

    preds = model.predict(X)                 # 1 = normal, -1 = anomaly
    scores = model.decision_function(X)      # lower = more anomalous

    scored = master.copy()
    scored["anomaly"] = preds
    scored["anomaly_score"] = scores
    anom = scored[scored["anomaly"] == -1]
    norm = scored[scored["anomaly"] == 1]

    anomaly_count = int((preds == -1).sum())
    metrics = {
        "anomaly_count": anomaly_count,
        "anomaly_percentage": round(anomaly_count / len(preds), 4),
        "mean_score": float(scores.mean()),
    }
    # separation metrics: how different are anomalies from normal orders?
    for col in ["payment_value", "freight_value", "delivery_days", "product_weight_g", "review_score"]:
        if col in scored.columns:
            metrics[f"{col}_normal"] = round(float(norm[col].mean()), 2)
            metrics[f"{col}_anomaly"] = round(float(anom[col].mean()), 2)
    mlflow.log_metrics(metrics)

    mlflow.sklearn.log_model(model, name="isolation_forest")
    run_id = run.info.run_id

print(f"MLflow run {run_id[:8]} logged to 'olist_anomaly_detection'")
print(f"Anomalies: {anomaly_count:,} ({metrics['anomaly_percentage']:.2%}) of {len(preds):,} orders")
print(f"Mean payment  normal R${metrics['payment_value_normal']:,.2f}  vs  anomaly R${metrics['payment_value_anomaly']:,.2f}")

## Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(norm["anomaly_score"], bins=60, color="#94A3B8", alpha=0.7, label="Normal")
ax.hist(anom["anomaly_score"], bins=60, color="#DC2626", alpha=0.7, label="Anomaly")
ax.set_title("Isolation Forest decision-function scores (lower = more anomalous)")
ax.set_xlabel("anomaly_score"); ax.set_ylabel("count"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Normal vs anomaly averages across business fields
compare_cols = ["payment_value", "freight_value", "delivery_days", "product_weight_g", "review_score"]
norm_means = [norm[c].mean() for c in compare_cols]
anom_means = [anom[c].mean() for c in compare_cols]

x = np.arange(len(compare_cols)); w = 0.38
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - w/2, norm_means, w, color="#94A3B8", label="Normal")
ax.bar(x + w/2, anom_means, w, color="#4F46E5", label="Anomaly")
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(compare_cols, rotation=20, ha="right")
ax.set_title("Normal vs anomaly — average by field (log scale)"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
anom["product_category"].value_counts().head(10).iloc[::-1].plot.barh(ax=ax1, color="#4F46E5")
ax1.set_title("Top product categories among anomalies")
anom["customer_state"].value_counts().head(10).iloc[::-1].plot.barh(ax=ax2, color="#0EA5E9")
ax2.set_title("Top customer states among anomalies")
plt.tight_layout(); plt.show()

## Compare your runs
Every time you edit `HYPERPARAMS` and re-run, a new run appears under the
`olist_anomaly_detection` experiment. Launch the UI:

```
.venv\Scripts\python -m mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Open http://localhost:5000 → experiment **olist_anomaly_detection** → select runs → **Compare**.
Watch `anomaly_percentage` (should track your `contamination`) and the `*_anomaly` vs `*_normal`
separation metrics — a good configuration flags orders that are clearly different from normal ones.